# Copy the PSU differentiable-model simulations from legacy

| v1 dataset (`public`) | what |
| --- | --- |
| `psu_dhbv_hf`, `psu_dprms_hf`, `psu_dsacsma_hf`, `psu_densemble_hf` | PSU simulated daily mean streamflow at 2,703 gages, 1992-10 to 2010-09 (legacy `secondary_timeseries`, 17.8 M rows each) |
| `psu_dhbv_hf_series` | the series table they share: keys are the gages' (`usgs-<site>`), each series' `location_key` the gage in `usgs_gages` |

Legacy keyed them by gage, as `usgs_daily_streamflow` is, so evaluations pair them on the
series' location, without a crosswalk:

```yaml
inputs:
  primary: {dataset: public.usgs_daily_streamflow}
  secondary: [{dataset: public.psu_dhbv_hf}]
pair: {on_attribute: location_key}
```

Legacy's `member` column holds the string `"None"` in every row: not copied.

**Run it as a platform admin, on use1.** Spark writes the rows straight into the `public`
Iceberg tables (small: one write per dataset). **Re-runnable:** registering skips datasets that
exist; series are added, never removed; each copy replaces the months it holds. Start with
**2. Count**: it writes nothing.

In [ ]:
import os

import pandas as pd
import requests
import teehr
from teehr.evaluation.spark_session_utils import create_spark_session

teehr.__version__

## 1. Settings

In [ ]:
#: Legacy configuration -> v1 dataset (the same names).
DATASETS = ["psu_dhbv_hf", "psu_dprms_hf", "psu_dsacsma_hf", "psu_densemble_hf"]
ANCHOR = DATASETS[0]  # makes the series table; the others share it
SERIES, LAYER = f"{ANCHOR}_series", "usgs_gages"
VARIABLE = "streamflow_daily_mean"
SOURCE = ("Penn State (MHPI) differentiable models, AORC forcing, Hydrofabric v2.2, 1980-1995 "
          "training; copied from the legacy TEEHR warehouse")

CONTROL_PLANE = os.environ.get(
    "CONTROL_PLANE_URL", "http://teehr-control-plane.teehr-hub.svc.cluster.local:8000"
)
VERIFY_TLS = "local.app.garden" not in CONTROL_PLANE  # KinD's certificate is self-signed
LEGACY, PUBLIC = "iceberg.teehr", "iceberg.public"

In [ ]:
# Local mode: 71 M rows in all.
spark = create_spark_session()
spark.conf.set("spark.sql.iceberg.check-nullability", "false")
spark.conf.set("spark.sql.session.timeZone", "UTC")

def access_token():
    """Your access token, refreshed when it's about to expire (JupyterHub sets it)."""
    try:
        from teehr.utils.keycloak_auth import refresh_user_token_from_env
    except ImportError:  # a kernel without the helper: the token as JupyterHub set it
        return os.environ["POLARIS_USER_TOKEN"]
    access, _refresh = refresh_user_token_from_env()
    return access


api = requests.Session()
api.verify = VERIFY_TLS


def call(method, path, **kwargs):
    resp = api.request(method, f"{CONTROL_PLANE}/v1{path}",
                       headers={"Authorization": f"Bearer {access_token()}"}, **kwargs)
    if resp.status_code >= 400 and resp.status_code != 404:
        raise RuntimeError(f"{method} {path}: HTTP {resp.status_code}: {resp.text[:1000]}")
    return resp


def exists(name):
    return call("GET", f"/projects/public/datasets/{name}").status_code == 200


def register(body):
    if exists(body["name"]):
        print(f"public.{body['name']}: exists")
        return
    call("POST", "/projects/public/datasets", json=body)
    print(f"public.{body['name']}: registered")


def month_range(column, month):
    """A time range filter for one UTC month (prunes the scan, unlike date_format)."""
    start = pd.Period(month, freq="M")
    return (f"{column} >= TIMESTAMP '{start.start_time:%Y-%m-%d} 00:00:00' "
            f"AND {column} < TIMESTAMP '{(start + 1).start_time:%Y-%m-%d} 00:00:00'")


def per_month(table, name, where="TRUE"):
    return spark.sql(f"""
        SELECT date_format(value_time, 'yyyy-MM') AS month, count(*) AS {name}
        FROM {table} WHERE {where} AND value IS NOT NULL GROUP BY 1
    """).toPandas()


me = call("GET", "/projects/public").json()
print(f"public: {me['status']}; your role there: {me.get('my_role')}")

## 2. Count (writes nothing)

In [ ]:
%%time
spark.sql(f"""
    SELECT configuration_name, unit_name, member, count(*) AS rows,
           count(DISTINCT location_id) AS gages, min(value_time) AS first, max(value_time) AS last,
           count_if(hour(value_time) <> 0) AS not_midnight
    FROM {LEGACY}.secondary_timeseries
    WHERE configuration_name IN ({", ".join(f"'{d}'" for d in DATASETS)})
      AND variable_name = '{VARIABLE}'
    GROUP BY 1, 2, 3 ORDER BY 1
""").show(truncate=False)
descriptions = {r.name: r.description for r in spark.sql(f"""
    SELECT name, description FROM {LEGACY}.configurations
    WHERE name IN ({", ".join(f"'{d}'" for d in DATASETS)})
""").collect()}
descriptions

## 3. Register (as you) and add the series

Simulated daily means at midnight UTC of their day (`mean`, `day_starting`); the anchor first,
which makes the series table. Each gage is a series whose location is itself in `usgs_gages`.

In [ ]:
for dataset in DATASETS:
    register({
        "type": "timeseries", "name": dataset, "role": "simulated", "variable": "streamflow",
        "statistic": "mean", "period": "day_starting", "location_layer": LAYER,
        **({} if dataset == ANCHOR else {"series_table": SERIES}),
        "source": SOURCE, "description": descriptions.get(dataset),
    })
spark.sql(f"""
    MERGE INTO {PUBLIC}.{SERIES} t
    USING (SELECT DISTINCT location_id AS series_key FROM {LEGACY}.secondary_timeseries
           WHERE configuration_name IN ({", ".join(f"'{d}'" for d in DATASETS)})
             AND variable_name = '{VARIABLE}') s
    ON t.series_key = s.series_key
    WHEN NOT MATCHED THEN INSERT (series_key, location_key, location_layer, period, resolution)
        VALUES (s.series_key, s.series_key, '{LAYER}', NULL, NULL)
""")
print(f"{SERIES}: {spark.table(f'{PUBLIC}.{SERIES}').count():,} series")

## 4. Copy

One write per dataset, replacing the months it holds. Values are kept as they are (doubles):
model output, not a gauge's rounded readings.

In [ ]:
%%time
for dataset in DATASETS:
    spark.sql(f"""
        SELECT location_id AS series_key, value_time, CAST(value AS DOUBLE) AS value
        FROM {LEGACY}.secondary_timeseries
        WHERE configuration_name = '{dataset}' AND variable_name = '{VARIABLE}'
          AND value IS NOT NULL
    """).writeTo(f"{PUBLIC}.{dataset}").overwritePartitions()
    print(f"{dataset}: copied")

## 5. Check: v1 against legacy, month by month

In [ ]:
%%time
for dataset in DATASETS:
    both = per_month(f"{LEGACY}.secondary_timeseries", "legacy",
                     f"configuration_name = '{dataset}' AND variable_name = '{VARIABLE}'").merge(
        per_month(f"{PUBLIC}.{dataset}", "v1"), on="month", how="outer").fillna(0)
    both["matches"] = both.legacy == both.v1
    print(f"{dataset}: {int(both.v1.sum()):,} rows; every month matches: {both.matches.all()}")
    display(both[~both.matches])